In [ ]:
import os, sys, mne, socket, time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn import plotting

from initialization import load_paths

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
subjID = 99
day = 3
p = load_paths(subjID, day)
montage = mne.channels.make_standard_montage("easycap-M1")

In [ ]:
# List all the files in the directory
f_list = os.listdir(p['EEGfiles'])
f_list = [f for f in f_list if f.endswith('.vhdr')]
if len(f_list) == 1:
    raw_data = mne.io.read_raw_brainvision(p['EEGfiles'] + '/' + f_list[0], preload=True)
    raw_data.drop_channels(['LM', 'RM'])
    raw_data.set_montage(montage)


In [ ]:
raw_data.compute_psd(fmax=55, average='mean', n_fft=2048).plot()
plt.show()

In [ ]:
raw_data = mne.add_reference_channels(raw_data, ref_channels='Cz')

In [ ]:
events, tah = mne.events_from_annotations(raw_data)
events_dict = {
    'BlockOnset': 1001,
    'BlockEnd': 7,
    'Fixation': 1,
    'Delay1': 2,
    'Delay2': 3,
    'Response': 4,
    'Feedback': 5,
    'ITI': 6,
}
trl_events = {
    'inPF': 11,
    'outPF': 10,
}

In [ ]:
block_onset_id = events_dict['BlockOnset']
block_end_id = events_dict['BlockEnd']
# Find the start and end events
start_events = events[events[:, 2] == block_onset_id]
end_events = events[events[:, 2] == block_end_id]

# Create custom epochs
block_epochs_list = []
for start_event in start_events:
    # Find the next end event after the current start event
    end_event = end_events[end_events[:, 0] > start_event[0]][0]

    # Define the time window for the block epoch
    tmin = (start_event[0] - raw_data.first_samp) / raw_data.info['sfreq']
    tmax = (end_event[0] - raw_data.first_samp) / raw_data.info['sfreq']

    # Create the block epoch
    block = raw_data.copy().crop(tmin, tmax)
    block_epochs_list.append(block)

# Concatenate all epochs
# epochs = mne.concatenate_epochs(epochs_list)

In [ ]:
# Apply a highpass filter to each epoch
block_epochs_list = [block.filter(l_freq=0.1, h_freq=None) for block in block_epochs_list]

In [ ]:
# Print duration of the epochs
total_dur = 0
for epoch in epochs_list:
    total_dur += epoch.times[-1]
    print(f'Epoch duration: {epoch.times[-1]:.2f} s')
print(f'Total duration: {total_dur:.2f} s')

In [ ]:
# Re-epoch data